# Evaluasi Data Uji

Notebook ini menilai model yang sudah dikunci pada data uji, sekali saja. Model tidak diubah berdasarkan hasil di sini.

**Model terkunci:** XGBoost, parameter default, tanpa pembobotan, 12 fitur.

**Pembanding:** XGBoost dengan 10 fitur tanpa `Status` dan `Complains`, dan `DummyClassifier` sebagai batas bawah.

**Keterbatasan:** Analisis mengikuti dokumentasi UCI yang menyatakan tiap baris mewakili satu pelanggan. Audit menemukan pola berulang per 50 baris, sehingga independensi antarbaris tidak terjamin dan skor model bisa terlalu optimistis. Data uji dibagi acak dan memuat pola yang sama dengan data latih, jadi skornya tidak boleh dibaca sebagai performa pada pelanggan baru.

## 1. Persiapan

In [1]:
from pathlib import Path

import numpy as np
import pandas as pd
from sklearn.dummy import DummyClassifier
from sklearn.metrics import average_precision_score, roc_auc_score
from sklearn.model_selection import StratifiedKFold, train_test_split
from sklearn.neighbors import NearestNeighbors
from sklearn.preprocessing import StandardScaler
from xgboost import XGBClassifier

SEED = 180915
N_BOOTSTRAP = 2000

# Path relatif terhadap root project
ROOT = Path.cwd().parent if Path.cwd().name == 'program' else Path.cwd()

In [2]:
df = pd.read_csv(ROOT / 'dataset_clean' / 'customer_churn_clean.csv')

TARGET = 'Churn'
FEATURES_ALL = [col for col in df.columns if col not in (TARGET, 'Age_Binning')]
FEATURES_REDUCED = [col for col in FEATURES_ALL if col not in ('Status', 'Complains')]
assert TARGET not in FEATURES_ALL and 'audit_group' not in FEATURES_ALL
assert len(FEATURES_ALL) == 12 and len(FEATURES_REDUCED) == 10

# Pembagian yang sama dengan notebook perbandingan model dan penyetelan
train_idx, test_idx = train_test_split(
    df.index, test_size=0.2, stratify=df[TARGET], random_state=SEED
)
assert not set(train_idx) & set(test_idx)
train, test = df.loc[train_idx], df.loc[test_idx]
y_train, y_test = train[TARGET], test[TARGET]
assert len(train) == 2280 and y_train.sum() == 357
assert len(test) == 570 and y_test.sum() == 89

print(f'Latih: {len(train):,} baris, {y_train.sum():,} churn')
print(f'Uji  : {len(test):,} baris, {y_test.sum():,} churn ({y_test.mean():.2%})')

Latih: 2,280 baris, 357 churn
Uji  : 570 baris, 89 churn (15.61%)


## 2. Melatih dan menilai

Semua model dilatih pada seluruh data latih, lalu dinilai pada data uji.

In [3]:
MODELS = {
    'XGBoost, 12 fitur (terkunci)': (XGBClassifier(random_state=SEED, eval_metric='logloss'), FEATURES_ALL),
    'XGBoost, 10 fitur (pembanding)': (XGBClassifier(random_state=SEED, eval_metric='logloss'), FEATURES_REDUCED),
    'DummyClassifier (batas bawah)': (DummyClassifier(strategy='prior'), FEATURES_ALL),
}

scores = {}
for name, (model, features) in MODELS.items():
    model.fit(train[features], y_train)
    scores[name] = model.predict_proba(test[features])[:, 1]

## 3. Skor dan ketidakpastian

Rentang memakai bootstrap persentil 95% pada baris uji. Rentang ini mengasumsikan baris independen, jadi bisa terlalu sempit. Rentang ini juga hanya menggambarkan variasi data uji, bukan performa pada pelanggan baru.

In [4]:
def bootstrap_interval(y_true, y_score, metric, seed=SEED, n_boot=N_BOOTSTRAP):
    rng = np.random.default_rng(seed)
    y_true, y_score = np.asarray(y_true), np.asarray(y_score)
    values = []
    while len(values) < n_boot:
        idx = rng.integers(0, len(y_true), len(y_true))
        # Sampel tanpa dua kelas tidak bisa dinilai
        if y_true[idx].min() == y_true[idx].max():
            continue
        values.append(metric(y_true[idx], y_score[idx]))
    return np.percentile(values, [2.5, 97.5])


def fmt(value, interval):
    return f'{value:.3f} ({interval[0]:.3f}–{interval[1]:.3f})'.replace('.', ',')


rows = []
for name, y_score in scores.items():
    pr = average_precision_score(y_test, y_score)
    roc = roc_auc_score(y_test, y_score)
    rows.append({
        'Model': name,
        'PR-AUC': fmt(pr, bootstrap_interval(y_test, y_score, average_precision_score)),
        'ROC-AUC': fmt(roc, bootstrap_interval(y_test, y_score, roc_auc_score)),
    })
print(pd.DataFrame(rows).set_index('Model').to_string())

                                             PR-AUC              ROC-AUC
Model                                                                   
XGBoost, 12 fitur (terkunci)    0,958 (0,927–0,979)  0,991 (0,983–0,996)
XGBoost, 10 fitur (pembanding)  0,940 (0,900–0,972)  0,980 (0,961–0,994)
DummyClassifier (batas bawah)   0,156 (0,126–0,186)  0,500 (0,500–0,500)


## 4. Pemeriksaan dan pembacaan

In [5]:
dummy_score = scores['DummyClassifier (batas bawah)']
assert abs(average_precision_score(y_test, dummy_score) - y_test.mean()) < 0.001
assert abs(roc_auc_score(y_test, dummy_score) - 0.5) < 0.001
print(f'Porsi churn data uji: {y_test.mean():.3f}')

# Skor CV acak untuk perbandingan, dari program/Tuning_XGBoost.ipynb (fold seed baru)
cv_pr_12 = 0.934
cv_pr_10 = 0.885
main_pr = average_precision_score(y_test, scores['XGBoost, 12 fitur (terkunci)'])
reduced_pr = average_precision_score(y_test, scores['XGBoost, 10 fitur (pembanding)'])
print(f'XGBoost 12 fitur: uji {main_pr:.3f}, CV acak {cv_pr_12:.3f}, selisih {main_pr - cv_pr_12:+.3f}')
print(f'XGBoost 10 fitur: uji {reduced_pr:.3f}, CV acak {cv_pr_10:.3f}, selisih {reduced_pr - cv_pr_10:+.3f}')
print(f'Dampak membuang Status dan Complains pada data uji: {main_pr - reduced_pr:+.3f}')

Porsi churn data uji: 0.156
XGBoost 12 fitur: uji 0.958, CV acak 0.934, selisih +0.024
XGBoost 10 fitur: uji 0.940, CV acak 0.885, selisih +0.055
Dampak membuang Status dan Complains pada data uji: +0.018


## 5. Mengapa skor data uji lebih tinggi dari CV acak

Skor data uji biasanya sama atau sedikit lebih rendah dari CV, jadi selisih ini diperiksa. Pemeriksaan hanya untuk memahami hasil, dan model tidak berubah. Ada tiga kemungkinan:

1. Model akhir dilatih dengan lebih banyak baris daripada model di dalam CV.
2. Pembagian data uji kebetulan menguntungkan.
3. Baris uji lebih mirip data latih daripada baris validasi di dalam CV.

In [6]:
# Kemungkinan 1: jumlah baris latih. Hanya memakai data latih
train_only = train.reset_index(drop=True)
y_only = train_only[TARGET]
FEATURE_SETS = {'12 fitur': FEATURES_ALL, '10 fitur': FEATURES_REDUCED}


def new_model():
    return XGBClassifier(random_state=SEED, eval_metric='logloss')


rows = []
for n_splits, n_repeats in ((5, 4), (10, 4), (20, 2)):
    row = {'Fold': n_splits, 'Baris latih per fold': int(len(train_only) * (n_splits - 1) / n_splits)}
    for name, features in FEATURE_SETS.items():
        repeat_scores = []
        for repeat in range(n_repeats):
            splitter = StratifiedKFold(n_splits=n_splits, shuffle=True, random_state=SEED + 50 + repeat)
            fold_scores = []
            for fit_idx, val_idx in splitter.split(train_only, y_only):
                model = new_model().fit(train_only[features].iloc[fit_idx], y_only.iloc[fit_idx])
                predicted = model.predict_proba(train_only[features].iloc[val_idx])[:, 1]
                fold_scores.append(average_precision_score(y_only.iloc[val_idx], predicted))
            repeat_scores.append(np.mean(fold_scores))
        row[f'PR-AUC {name}'] = round(np.mean(repeat_scores), 3)
    rows.append(row)
print(pd.DataFrame(rows).to_string(index=False))

 Fold  Baris latih per fold  PR-AUC 12 fitur  PR-AUC 10 fitur
    5                  1824            0.935            0.892
   10                  2052            0.937            0.899
   20                  2166            0.941            0.904


In [7]:
# Kemungkinan 2: variasi antar-pembagian 80:20. Hanya memakai data latih
split_scores = {name: [] for name in FEATURE_SETS}
for offset in range(10):
    fit_idx, val_idx = train_test_split(
        train_only.index, test_size=0.2, stratify=y_only, random_state=SEED + 100 + offset
    )
    for name, features in FEATURE_SETS.items():
        model = new_model().fit(train_only.loc[fit_idx, features], y_only.loc[fit_idx])
        predicted = model.predict_proba(train_only.loc[val_idx, features])[:, 1]
        split_scores[name].append(average_precision_score(y_only.loc[val_idx], predicted))

for name, values in split_scores.items():
    print(f'{name}: PR-AUC {min(values):.3f} sampai {max(values):.3f} pada 10 pembagian')

12 fitur: PR-AUC 0.917 sampai 0.942 pada 10 pembagian
10 fitur: PR-AUC 0.863 sampai 0.933 pada 10 pembagian


In [8]:
# Kemungkinan 3: kemiripan dengan data latih, dihitung pada tetangga terdekat
def share_near_identical(query, reference, features, max_diff=1):
    scaler = StandardScaler().fit(reference[features])
    neighbors = NearestNeighbors(n_neighbors=1).fit(scaler.transform(reference[features]))
    _, indices = neighbors.kneighbors(scaler.transform(query[features]))
    different = (query[features].values != reference[features].values[indices[:, 0]]).sum(axis=1)
    return (different <= max_diff).mean()


fit_idx, val_idx = train_test_split(
    train_only.index, test_size=0.2, stratify=y_only, random_state=SEED + 50
)
share_test = share_near_identical(test, train, FEATURES_ALL)
share_val = share_near_identical(train_only.loc[val_idx], train_only.loc[fit_idx], FEATURES_ALL)
print(f'Baris uji yang beda maksimal 1 kolom dari data latih: {share_test:.1%}')
print(f'Baris validasi yang beda maksimal 1 kolom dari data latih: {share_val:.1%}')

Baris uji yang beda maksimal 1 kolom dari data latih: 3.5%
Baris validasi yang beda maksimal 1 kolom dari data latih: 3.3%


### Pembacaan pemeriksaan

- Jumlah baris latih menjelaskan sebagian kenaikan, karena skor CV naik saat baris latih per fold bertambah.
- Pembagian yang menguntungkan masih mungkin, tetapi pembagian di dalam data latih memakai baris latih lebih sedikit, jadi rentangnya tidak bisa dibandingkan langsung dengan skor data uji.
- Kemiripan dihitung dari tetangga terdekat, dan tidak terlihat beda berarti antara baris uji dan baris validasi.
- Penyebab pastinya belum terbukti. CV acak tetap menjadi angka yang lebih stabil, karena berasal dari banyak pembagian.

## Cara membaca hasil

- Skor data uji yang mirip dengan CV acak tidak membuktikan model siap untuk pelanggan baru, karena data uji dibagi dengan cara yang sama.
- Semua hubungan bersifat asosiatif, bukan kausal.
- Threshold, precision, recall, dan F1 dipilih dan dilaporkan di Fase 3.